In [0]:
%sql
SELECT COUNT(*) 
FROM workspace.default.gold_traffic_weather_hourly;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.gold_road_network_summary;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.gold_accident_summary;

In [0]:
%sql
SELECT COUNT(*) FROM workspace.default.gold_transport_summary;

In [0]:
# UrbanPulse — Smart City Mobility Intelligence Platform
# Data Engineering Architecture Diagram (Bronze → Silver → Gold → Dashboard)

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

INK, MUTED, ARROW = "#16233A", "#5B6B82", "#43536B"
SRC_H, SRC_F, SRC_E = "#3D5A80", "#EDF1F8", "#9FB0CC"
BRZ_H, BRZ_F, BRZ_E = "#A97A1F", "#F8ECCE", "#D3A84C"
SLV_H, SLV_F, SLV_E = "#5D6E88", "#E9EEF4", "#A3B2C9"
GLD_H, GLD_F, GLD_E = "#8A6D1C", "#FAEFC8", "#C9AC5B"
VIZ_H, VIZ_F, VIZ_E = "#2F5BC7", "#E6EFFF", "#7FA0E8"

fig, ax = plt.subplots(figsize=(13.5, 19), dpi=140)
fig.patch.set_facecolor("white")
ax.set_xlim(0, 100)
ax.set_ylim(-4, 110)
ax.axis("off")
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)


def box(cx, cy, w, h, fill, edge, lw=1.3, ls="-"):
    ax.add_patch(FancyBboxPatch((cx - w / 2, cy - h / 2), w, h,
                                boxstyle="round,pad=0,rounding_size=0.9",
                                facecolor=fill, edgecolor=edge,
                                linewidth=lw, linestyle=ls))


def label(cx, cy, s, size=10, color=INK, weight="normal", mono=False, italic=False):
    ax.text(cx, cy, s, ha="center", va="center", fontsize=size, color=color,
            fontweight=weight, family="monospace" if mono else "sans-serif",
            style="italic" if italic else "normal")


def header(x, y, s, color):
    ax.text(x, y, s, ha="left", va="center", fontsize=12.5, color=color, fontweight="bold")


def badge(x_right, y, s, color):
    ax.text(x_right, y, s, ha="right", va="center", fontsize=9, color=color,
            fontweight="bold", bbox=dict(boxstyle="round,pad=0.45",
                                         facecolor="white", edgecolor=color, linewidth=1.2))


def arrow(x0, y0, x1, y1):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>",
                                mutation_scale=24, linewidth=2.4, color=ARROW,
                                capstyle="round"))


def chip_row(names, cy, h, w, g, x0, fill, edge, size):
    for i, nm in enumerate(names):
        cx = x0 + w / 2 + i * (w + g)
        box(cx, cy, w, h, fill, edge, lw=1.2)
        label(cx, cy, nm, size, mono=True)


# ---------------------------------------------------------------- title
label(50, 106.2, "UrbanPulse — Smart City Mobility Intelligence Platform", 21, INK, "bold")
label(50, 102.8, "Data Engineering Architecture · Medallion Pipeline on Databricks", 12, MUTED)

# ---------------------------------------------------------------- 1 · data sources
box(50, 93.25, 94, 13.5, "#FAFBFD", SRC_E, lw=1.6)
header(5.2, 97.6, "1 · DATA SOURCES", SRC_H)
sources = [
    ("OpenStreetMap", "Delhi road network"),
    ("Traffic Probe Dataset", "Delhi traffic observations"),
    ("Open-Meteo", "Historical weather data"),
    ("Road Accident Statistics", "Delhi crash records"),
    ("GTFS", "Routes · trips · stops · schedules"),
]
for i, (nm, ds) in enumerate(sources):
    cx = 4 + 16.4 / 2 + i * (16.4 + 2.5)
    box(cx, 91.6, 16.4, 6.6, SRC_F, SRC_E, lw=1.4)
    label(cx, 92.7, nm, 10.2, INK, "bold")
    label(cx, 90.4, ds, 8.4, MUTED)

for x in (25, 50, 75):
    arrow(x, 86.3, x, 82.9)

# ---------------------------------------------------------------- 2 · bronze
box(50, 76.75, 94, 11.5, "#FAFBFD", BRZ_E, lw=1.6)
header(5.2, 79.8, "2 · BRONZE LAYER — RAW DATA", BRZ_H)
badge(94.8, 79.8, "Unity Catalog Volumes", BRZ_H)
box(50, 74.9, 88, 5.6, BRZ_F, BRZ_E)
label(50, 76.1, "Ingested source datasets stored as raw files in Unity Catalog Volumes", 10.2, INK, "bold")
label(50, 73.6, "Original raw data preserved for future reprocessing", 9.2, MUTED)

arrow(50, 70.7, 50, 66.9)
label(58, 68.8, "PySpark processing", 9.2, MUTED, italic=True)

# ---------------------------------------------------------------- 3 · silver
box(50, 52.75, 94, 27.5, "#FAFBFD", SLV_E, lw=1.6)
header(5.2, 63.8, "3 · SILVER LAYER — CLEANED & TRANSFORMED DATA", SLV_H)
badge(94.8, 63.8, "PySpark · Delta Lake", SLV_H)
box(50, 60.1, 70, 4.2, "white", SLV_E, lw=1.1, ls="--")
label(50, 60.1, "PySpark — data cleaning · standardization · transformation", 9.6, SLV_H, "bold")

silver_row_1 = ["silver_traffic", "silver_traffic_hourly", "silver_weather",
                "silver_road_edges", "silver_road_nodes"]
silver_row_2 = ["silver_crash_statistics", "silver_stops", "silver_routes",
                "silver_trips", "silver_stop_times"]
chip_row(silver_row_1, 55.2, 5.6, 16.5, 2.5, 4.5, SLV_F, SLV_E, 9.3)
chip_row(silver_row_2, 48.2, 5.6, 16.5, 2.5, 4.5, SLV_F, SLV_E, 9.3)

arrow(50, 38.8, 50, 34.9)
label(58, 36.8, "aggregation & joins", 9.2, MUTED, italic=True)

# ---------------------------------------------------------------- 4 · gold
box(50, 28.75, 94, 11.5, "#FAFBFD", GLD_E, lw=1.6)
header(5.2, 31.9, "4 · GOLD LAYER — ANALYTICS-READY DATA", GLD_H)
badge(94.8, 31.9, "Delta Lake", GLD_H)
gold_tables = ["gold_traffic_weather_hourly", "gold_road_network_summary",
               "gold_accident_summary", "gold_transport_summary"]
chip_row(gold_tables, 26.9, 5.8, 20.8, 2.6, 4.5, GLD_F, GLD_E, 9.3)

arrow(50, 22.8, 50, 18.9)
label(57.5, 20.8, "SQL queries", 9.2, MUTED, italic=True)

# ---------------------------------------------------------------- 5 · visualization
box(50, 10, 94, 17, "#FAFBFD", VIZ_E, lw=1.6)
header(5.2, 15.8, "5 · VISUALIZATION LAYER", VIZ_H)
badge(94.8, 15.8, "Databricks AI/BI", VIZ_H)
box(50, 12.1, 46, 4.6, "#2F5BC7", "#2F5BC7")
label(50, 12.1, "Databricks AI/BI Dashboard", 12, "white", "bold")

viz_row_1 = ["KPI cards", "Traffic activity analysis", "Weather impact exploration"]
viz_row_2 = ["Road infrastructure analysis", "Accident trends", "Public transport analysis"]
for row, cy in ((viz_row_1, 7.7), (viz_row_2, 4.0)):
    for i, nm in enumerate(row):
        cx = 4.5 + 28.3 / 2 + i * (28.3 + 3.2)
        box(cx, cy, 28.3, 3.6, VIZ_F, VIZ_E, lw=1.2)
        label(cx, cy, nm, 9.4)

# ---------------------------------------------------------------- footer
label(50, -1.6, "Databricks Lakehouse Platform — PySpark · Delta Lake · Unity Catalog · AI/BI Dashboards", 10.5, MUTED)

plt.show()